# WATERWEAVE: Quantum Water-Allocation Decision Support
**Qiskit Fall Fest 2026 — Track 2: Open-Domain Optimization & Real-World Solvers**

> **Project Overview:**
> WATERWEAVE models municipal water rationing under drought scarcity as a constrained binary optimization problem. It formulates a multi-slot, multi-neighborhood trilemma between priority weighting, supply constraints, and service equity. This notebook walks through the problem formulation, classical baselines, Qiskit QAOA execution, telemetry inspection, and benchmark findings.

> **Important Disclaimer:**
> All data in this notebook is **strictly synthetic**. QAOA is simulated classically via Qiskit's `StatevectorSampler`. No quantum advantage is claimed; classical exact enumeration solves this small instance in milliseconds.

## 1. Synthetic Water-Allocation Scenario

We consider 4 neighborhoods ($N_1$ Hillview, $N_2$ Riverside, $N_3$ Market, $N_4$ Lakeside) across 2 delivery slots (`morning`, `evening`).
Total municipal demand is **7 units**, while total available supply is only **4 units** (2 morning, 2 evening). Because demand strictly exceeds supply, trade-offs between neighborhood priority and equitable access are unavoidable.

In [1]:
import sys, os
if os.path.exists('src'):
    sys.path.insert(0, 'src')
elif os.path.exists('../src'):
    sys.path.insert(0, '../src')

from waterweave.scenarios import default_scenario
import pandas as pd

scenario = default_scenario(seed=42)
print(f'Scenario: {scenario.name}')
print(f'Delivery Slots: {scenario.slots}')
print(f'Reservoir Supply: {scenario.supply_by_slot}')

rows = []
for n in scenario.neighborhoods:
    rows.append({
        'ID': n.id,
        'Neighborhood': n.name,
        'Morning Demand': n.demand_by_slot.get('morning', 0),
        'Evening Demand': n.demand_by_slot.get('evening', 0),
        'Total Demand': sum(n.demand_by_slot.values()),
        'Priority Weight': f'{n.priority_weight:.1f}x',
        'Storage Cap (units)': n.storage_capacity_units,
    })
pd.DataFrame(rows)

Scenario: synthetic-4x2
Delivery Slots: ('morning', 'evening')
Reservoir Supply: {'morning': 2, 'evening': 2}


,ID,Neighborhood,Morning Demand,Evening Demand,Total Demand,Priority Weight,Storage Cap (units)
0,N1,Hillview (synthetic),1,1,2,1.5x,2
1,N2,Riverside (synthetic),1,1,2,1.0x,1
2,N3,Market (synthetic),1,0,1,1.0x,2
3,N4,Lakeside (synthetic),1,1,2,2.0x,2


## 2. Binary Optimization Formulation & QUBO/Ising Mapping

The objective minimizes:
$$\min_x F(x) = \text{Weighted Shortage}(x) + \alpha \cdot \text{Disparity}(x) + P \cdot \text{Violations}(x)$$

1. **Primary Decision Variables**: $x_{i,t} \in \{0, 1\}$ created only where demand exists ($d_{i,t} \ge 1$). Here, $K=7$ binary decision variables.
2. **Slack Variable Encoding**: Reservoir supply ($\sum x \le 2$) and cistern storage ($\sum x \le 1$) inequalities are converted into quadratic penalties using unary slack variables:
   $$P \left( \sum x + \sum s - B \right)^2$$
   This adds 5 slack bits, requiring a total of **12 qubits**.
3. **Ising Mapping**: $x_k = \frac{I - Z_k}{2}$ transforms the QUBO into diagonal cost Hamiltonian $H_C = \sum h_k Z_k + \sum J_{kl} Z_k Z_l + c_0 I$.

In [2]:
from waterweave.formulation import build_model

model = build_model(scenario, fairness_weight=2.0)
print(f'Primary Decision Variables (K): {model.n_decision}')
print(f'Slack Variables:                {model.n_slack}')
print(f'Total Problem Qubits:           {model.n_vars}')
print(f'Ising Offset Energy:           {model.ising_offset:.4f}')
print(f'Active Two-Qubit Couplings:     {(abs(model.ising_J) > 1e-9).sum() // 2} pairs')
print(f'Max Coupling Strength |J_kl|:   {abs(model.ising_J).max():.4f}')

Primary Decision Variables (K): 7
Slack Variables:                5
Total Problem Qubits:           12
Ising Offset Energy:           83.7500
Active Two-Qubit Couplings:     19 pairs
Max Coupling Strength |J_kl|:   8.2500


## 3. Classical Baselines: Greedy vs. Exact

We evaluate two classical benchmarks:
* **Greedy Baseline**: An objective-aware heuristic with deterministic 1-opt local search. It avoids the naive trap of pure priority-first allocation (which produced objective 11.0, worse than allocating nothing).
* **Exact Classical**: Exhaustive integer enumeration over all $2^7 = 128$ decision patterns. Finds the certified global optimum (objective $6.0$) in milliseconds.

In [3]:
from waterweave.solvers import solve_greedy, solve_exact

greedy_res = solve_greedy(scenario, fairness_weight=2.0)
exact_res = solve_exact(scenario, fairness_weight=2.0)

print('--- Greedy Heuristic ---')
print(f'Objective:      {greedy_res.objective_value:.4f}')
print(f'Feasible:       {greedy_res.feasible}')
print(f'Fairness Score: {greedy_res.metrics["fairness_score"]:.4f}')
print(f'Disparity:      {greedy_res.metrics["disparity"]:.4f}')
print(f'Runtime:        {greedy_res.runtime_seconds * 1000:.3f} ms')

print('--- Exact Classical (Optimal) ---')
print(f'Objective:      {exact_res.objective_value:.4f}')
print(f'Feasible:       {exact_res.feasible}')
print(f'Fairness Score: {exact_res.metrics["fairness_score"]:.4f}')
print(f'Disparity:      {exact_res.metrics["disparity"]:.4f}')
print(f'Runtime:        {exact_res.runtime_seconds * 1000:.3f} ms')

--- Greedy Heuristic ---
Objective:      7.0000
Feasible:       True
Fairness Score: 0.5000
Disparity:      0.7500
Runtime:        0.983 ms
--- Exact Classical (Optimal) ---
Objective:      6.0000
Feasible:       True
Fairness Score: 0.5000
Disparity:      0.7500
Runtime:        3.395 ms


## 4. QAOA Circuit Construction & Execution

The QAOA ansatz applies alternating unitaries on 12 qubits:
$$|\psi(\gamma, \beta)\rangle = \prod_{l=1}^p \left[ e^{-i \beta_l H_M} e^{-i \gamma_l H_C} \right] |+\rangle^{\otimes n}$$

1. **Cost Layer**: $R_{ZZ}(2\gamma J_{kl})$ two-qubit interactions and $R_Z(2\gamma h_k)$ single-qubit rotations.
2. **Mixer Layer**: $R_X(2\beta)$ transverse field rotations.
3. **Optimization**: Warm-start grid search (36 evaluations) followed by SciPy COBYLA (25 evaluations).
4. **Sampling & Post-Selection**: Measured across 1,024 shots with Qiskit's `StatevectorSampler`. The best feasible bitstring is returned.

In [4]:
from waterweave.solvers import solve_qaoa, _build_qaoa_circuit

qaoa_res = solve_qaoa(scenario, p=1, shots=1024, maxiter=25, seed=42)
meta = qaoa_res.metadata

print('--- QAOA Solver Execution Telemetry ---')
print(f'Objective Value:         {qaoa_res.objective_value:.4f}')
print(f'Feasible:                {qaoa_res.feasible}')
print(f'Circuit Width:           {meta["num_qubits"]} qubits')
print(f'Circuit Depth:           {meta["circuit_depth"]}')
print(f'Optimal Gamma (Cost):    {[round(g, 4) for g in meta["optimal_gammas"]]}')
print(f'Optimal Beta (Mixer):    {[round(b, 4) for b in meta["optimal_betas"]]}')
print(f'Feasible Sample Ratio:   {meta["feasible_samples_ratio"] * 100:.1f}%')
print(f'Selected Bitstring:      {meta["best_bitstring"]}')
print(f'Modal Bitstring:         {meta["modal_bitstring"]}')
print(f'Simulation Runtime:      {qaoa_res.runtime_seconds:.4f} s')

# Reconstruct circuit for gate counts
qc = _build_qaoa_circuit(model, meta['optimal_gammas'], meta['optimal_betas'])
print(f'Circuit Gate Breakdown:  {dict(qc.count_ops())}')

--- QAOA Solver Execution Telemetry ---
Objective Value:         6.0000
Feasible:                True
Circuit Width:           12 qubits
Circuit Depth:           17
Optimal Gamma (Cost):    [0.4037]
Optimal Beta (Mixer):    [0.3971]
Feasible Sample Ratio:   74.4%
Selected Bitstring:      000001010110
Modal Bitstring:         101100010000
Simulation Runtime:      0.3264 s
Circuit Gate Breakdown:  {'rzz': 39, 'h': 12, 'rz': 12, 'rx': 12}


## 5. Comparative Evaluation: Greedy vs. Exact vs. QAOA

We summarize the performance of all three solvers on the shared synthetic scenario.
Notice the crucial mathematical separation between **Fairness Score** (range-based equality, higher is better) and **Disparity** (pairwise quadratic variance, minimized by the QUBO, lower is better).

In [5]:
results = [greedy_res, exact_res, qaoa_res]
table = []
for r in results:
    m = r.metrics
    table.append({
        'Solver': r.solver_name,
        'Feasible': 'Yes' if r.feasible else 'No',
        'Objective Value': f'{r.objective_value:.2f}',
        'Weighted Shortage': f'{m.get("weighted_shortage", 0):.2f}',
        'Service Level (%)': f'{m.get("service_level", 0)*100:.1f}%',
        'Fairness Score': f'{m.get("fairness_score", 0):.3f}',
        'Disparity (QUBO)': f'{m.get("disparity", 0):.3f}',
        'Runtime (s)': f'{r.runtime_seconds:.4f}',
    })
df_comp = pd.DataFrame(table)
df_comp

,Solver,Feasible,Objective Value,Weighted Shortage,Service Level (%),Fairness Score,Disparity (QUBO),Runtime (s)
0,Greedy (Priority-First),Yes,7.00,5.50,42.9%,0.500,0.750,0.0010
1,Exact Classical (Optimal),Yes,6.00,4.50,57.1%,0.500,0.750,0.0034
2,QAOA (Quantum Heuristic),Yes,6.00,4.50,57.1%,0.500,0.750,0.3264


## 6. Performance, Results & Critical Limitations

### Key Experimental Observations
1. **Solution Quality**: QAOA successfully finds a feasible allocation achieving the global optimum objective ($6.0$) in 80% to 100% of runs across depths $p \in \{1, 2, 3\}$.
2. **Classical Post-Selection**: The modal (most frequent) sampled bitstring is occasionally infeasible. The reported solution relies on classical post-selection of the best feasible candidate from the 1,024 sampled bitstrings.
3. **Uniform Random Control**: On 128 decision states, uniform random sampling also draws the optimal state in 15 of 15 runs. Best-of-1,024 sampling cannot separate QAOA from random search on this small instance.

### Scientific Limitations (No Quantum Advantage Claimed)
* **Simulation Overhead**: Exact classical enumeration evaluates all states in $3.5\text{ ms}$. QAOA classical simulation requires $\approx 350\text{ ms}$.
* **Hardware Noise**: Executing on real physical quantum processors (QPUs) would introduce gate infidelities and decoherence, requiring error mitigation and higher shot budgets.
* **Synthetic Data**: The problem instances demonstrate mathematical formulation mechanics and are not operational municipal water controllers.

In [6]:
# Load and inspect committed benchmark provenance
import os, json
bench_path = 'benchmark_results/benchmark_default.json' if os.path.exists('benchmark_results/benchmark_default.json') else '../benchmark_results/benchmark_default.json'
with open(bench_path, 'r', encoding='utf-8') as f:
    bench_data = json.load(f)

meta = bench_data['meta']
prov = meta['provenance']
print('Benchmark Provenance:')
print(f'  Engine Commit:    {prov["engine_git_commit"][:8]}')
print(f'  Python Version:   {prov["python"]}')
print(f'  Qiskit Version:   {prov["qiskit"]}')
print(f'  Audit Checks:     {len(bench_data["audit"]["checks"])} passed')
for c in bench_data['audit']['checks']:
    print(f'    - [{c["status"]}] {c["name"]}')

Benchmark Provenance:
  Engine Commit:    f5fe6b0f
  Python Version:   3.12.0
  Qiskit Version:   2.5.2
  Audit Checks:     6 passed
    - [pass] penalty_dominance
    - [pass] variable_order_matches
    - [pass] engine_objective_matches_independent
    - [pass] qubo_min_over_slack_equals_objective
    - [pass] ising_diagonal_equals_qubo_energy
    - [pass] independent_qaoa_simulation_matches_engine_circuit
